In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Adjust display settings for wider output
pd.set_option('display.width', 300)  # Increase the display width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 10)  # Limit rows displayed for clarity


generation = pd.read_csv("/Users/olaoluwatunmise/Germany-Energy-Price-Forecast/PowerCast-Datasets/Generation/Actual_generation_202301010000_202503050000_Quarterhour.csv", delimiter=';')

print(generation.head())

             Start date              End date Biomass [MWh] Original resolutions Hydropower [MWh] Original resolutions Wind offshore [MWh] Original resolutions Wind onshore [MWh] Original resolutions Photovoltaics [MWh] Original resolutions Other renewable [MWh] Original resolutions  \
0  Jan 1, 2023 12:00 AM  Jan 1, 2023 12:15 AM                           1,006.25                                319.75                                   684.25                                7,145.75                                     0.50                                      32.25   
1  Jan 1, 2023 12:15 AM  Jan 1, 2023 12:30 AM                           1,003.50                                317.25                                   743.50                                7,158.25                                     0.25                                      32.25   
2  Jan 1, 2023 12:30 AM  Jan 1, 2023 12:45 AM                           1,003.00                                 317.0                     

/var/folders/9f/dspzp56j3pjcftph22hs9gw40000gn/T/ipykernel_6004/3205126283.py:13: DtypeWarning: Columns (3,7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  generation = pd.read_csv("/Users/olaoluwatunmise/Germany-Energy-Price-Forecast/PowerCast-Datasets/Generation/Actual_generation_202301010000_202503050000_Quarterhour.csv", delimiter=';')


## Exploratory Data Analysis

In [39]:
# Clean the column names
generation.columns = generation.columns.str.replace(r'[^\w\s]', '', regex=True)  # Remove special characters
generation.columns = generation.columns.str.strip()  # Remove leading/trailing spaces
generation.columns = generation.columns.str.replace('Original resolutions', '', regex=False)  # Remove redundant words
# Remove "MWh" from the column names
generation.columns = generation.columns.str.replace(' MWh ', '', regex=False)
# Replace spaces between words with underscores for consistency
generation.columns = generation.columns.str.replace(' ', '_', regex=False)

# Replace multiple consecutive underscores with a single one (in case multiple spaces exist)
generation.columns = generation.columns.str.replace('_+', '_', regex=True)

# Ensure all column names are lowercase (optional, for consistency)
generation.columns = generation.columns.str.lower()

#new_consumption = consumption

print(generation.columns)
print(generation)

Index(['start_date', 'end_date', 'biomass', 'hydropower', 'wind_offshore', 'wind_onshore', 'photovoltaics', 'other_renewable', 'nuclear', 'lignite', 'hard_coal', 'fossil_gas', 'hydro_pumped_storage', 'other_conventional'], dtype='object')
                 start_date              end_date   biomass hydropower wind_offshore wind_onshore photovoltaics other_renewable nuclear lignite hard_coal fossil_gas hydro_pumped_storage other_conventional
0      Jan 1, 2023 12:00 AM  Jan 1, 2023 12:15 AM  1,006.25     319.75        684.25     7,145.75          0.50           32.25  615.25  962.75    517.00     453.75                31.75             307.25
1      Jan 1, 2023 12:15 AM  Jan 1, 2023 12:30 AM  1,003.50     317.25        743.50     7,158.25          0.25           32.25  614.75  963.25    518.00     453.50                45.00             307.25
2      Jan 1, 2023 12:30 AM  Jan 1, 2023 12:45 AM  1,003.00      317.0        817.00     7,302.25          0.25            32.5  615.00  966.50   

In [40]:
print(generation.isna().sum())

print(generation.describe())

print(generation.dtypes)

start_date              0
end_date                0
biomass                 0
hydropower              0
wind_offshore           0
                       ..
lignite                 0
hard_coal               0
fossil_gas              0
hydro_pumped_storage    0
other_conventional      0
Length: 14, dtype: int64
                  start_date              end_date   biomass  hydropower wind_offshore wind_onshore photovoltaics  other_renewable nuclear lignite hard_coal fossil_gas hydro_pumped_storage  other_conventional
count                  76224                 76224     76224    76224.00         76224        76224         76224          76224.0   76224   76224     76224      76224                76224             76224.0
unique                 76216                 76214      1707     3274.00          7125        30194         21909            193.0     806   13414     11623      14091                 6320              2031.0
top     Oct 29, 2023 2:30 AM  Oct 29, 2023 2:45 AM  1,063.00  

In [41]:
# Load raw data
#consumption = pd.read_csv("Datasets/Actual_consumption_Quarterhour.csv")
# print("Rows with '-' in total_grid_load:", (generation["total_grid_load"] == "-").sum())
# print("Rows with '-' in residual_load:", (generation["residual_load"] == "-").sum())
# print("Rows with '-' in hydro_pumped_storage:", (generation["hydro_pumped_storage"] == "-").sum())

generation.replace("-", np.nan, inplace=True)  # Ensure '-' is NaN
#consumption["total_grid_load"] = pd.to_numeric(consumption["total_grid_load"], errors="coerce")

In [42]:
print(generation.dtypes)
print(generation.columns)
print(generation.isna().sum())


start_date              object
end_date                object
biomass                 object
hydropower              object
wind_offshore           object
                         ...  
lignite                 object
hard_coal               object
fossil_gas              object
hydro_pumped_storage    object
other_conventional      object
Length: 14, dtype: object
Index(['start_date', 'end_date', 'biomass', 'hydropower', 'wind_offshore', 'wind_onshore', 'photovoltaics', 'other_renewable', 'nuclear', 'lignite', 'hard_coal', 'fossil_gas', 'hydro_pumped_storage', 'other_conventional'], dtype='object')
start_date               0
end_date                 0
biomass                 17
hydropower              17
wind_offshore           16
                        ..
lignite                 17
hard_coal               17
fossil_gas              17
hydro_pumped_storage    17
other_conventional      17
Length: 14, dtype: int64


In [43]:
# 1. Convert "Start date" to datetime (handle errors)
generation["start_date"] = pd.to_datetime(
    generation["start_date"], 
    format="%b %d, %Y %I:%M %p",  # Add explicit format to avoid warnings
    errors='coerce'
)

# Drop "End date" column early
generation.drop(columns=["end_date"], inplace=True)

# Set datetime column as index FIRST
generation.set_index("start_date", inplace=True)


# Remove commas and convert to float for specified columns
for col in ['biomass', 'hydropower', 'wind_offshore', 'wind_onshore', 'photovoltaics', 'other_renewable', 'nuclear', 'lignite', 'hard_coal', 'fossil_gas', 'hydro_pumped_storage', 'other_conventional']:
    # Replace commas and convert to float, assuming it's one number with commas as separators
    generation[col] = generation[col].str.replace(',', '').astype(float)

#Aggregate Duplicates Before Resampling
generation = generation.groupby(generation.index).mean()  # or .sum(), .max()

# Resample using the DatetimeIndex
generation_hourly = generation.resample('1h').sum()


# Reset index if needed (optional)
generation_hourly.reset_index(inplace=True)

print(generation_hourly.head())

           start_date  biomass  hydropower  wind_offshore  wind_onshore  photovoltaics  other_renewable  nuclear  lignite  hard_coal  fossil_gas  hydro_pumped_storage  other_conventional
0 2023-01-01 00:00:00  4014.25         0.0        3059.25      28710.50           1.25              0.0  2459.50  3859.25    2067.50     1817.75                125.25                 0.0
1 2023-01-01 01:00:00  3993.25         0.0        3586.00      29305.00           1.00              0.0  2458.75  3866.50    2052.00     1663.25                302.50                 0.0
2 2023-01-01 02:00:00  3967.25         0.0        3842.25      29266.00           1.25              0.0  2459.75  3860.25    2034.25     1666.75                141.50                 0.0
3 2023-01-01 03:00:00  3973.00         0.0        3463.25      27008.50           1.00              0.0  2460.50  3864.75    2037.00     1659.50                 96.25                 0.0
4 2023-01-01 04:00:00  3996.50         0.0        3462.25      26

In [44]:
print(generation_hourly.describe())

                start_date       biomass    hydropower  wind_offshore  wind_onshore  photovoltaics  other_renewable       nuclear       lignite     hard_coal    fossil_gas  hydro_pumped_storage  other_conventional
count                19056  19056.000000  19056.000000   19056.000000  19056.000000   19056.000000     19056.000000  19056.000000  19056.000000  19056.000000  19056.000000          19056.000000        19056.000000
mean   2024-02-01 23:30:00   4194.895105    200.267094    2849.774041  13189.565412    6531.017317        15.628306    353.733168   8592.694506   3956.218383   6594.062152           1168.606233          177.926952
min    2023-01-01 00:00:00      0.000000      0.000000       0.000000      0.000000       0.000000         0.000000      0.000000      0.000000      0.000000      0.000000              0.000000            0.000000
25%    2023-07-18 11:45:00   3977.000000      0.000000    1139.687500   4913.875000       2.500000         0.000000      0.000000   5547.875000 